# 35. Neural Networks: Recurrent Neural Networks (RNNs) and LSTM

## Algorithm Category
**Type**: Neural Networks - Deep Learning  
**Complexity**: High  
**Use Case**: Sequence modeling, time series prediction, natural language processing

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand RNNs and their architecture
- Implement RNNs and LSTMs using PyTorch
- Understand vanishing/exploding gradient problems
- Apply LSTMs to sequence prediction tasks
- Visualize hidden states and predictions
- Compare RNN vs LSTM performance

## Historical Context

RNNs were developed in the 1980s:
- Rumelhart, D.E., et al. (1986): "Learning representations by back-propagating errors"
- LSTM introduced by Hochreiter & Schmidhuber (1997)
- Foundation for modern sequence modeling

**Key Papers/References:**
- Hochreiter, S., & Schmidhuber, J. (1997). "Long short-term memory"
- Rumelhart, D.E., et al. (1986). "Learning representations by back-propagating errors"

## When to Use RNNs and LSTMs

RNNs/LSTMs are appropriate when:
- Working with sequential data
- Time series prediction
- Natural language processing
- Speech recognition
- When order matters in data
- Need to remember past information

## Theory & Mechanics

### Mathematical Foundation

**RNN Hidden State:**
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

**RNN Output:**
$$y_t = W_{hy} h_t + b_y$$

**LSTM Cell:**
- **Forget gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
- **Input gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
- **Cell state**: $C_t = f_t * C_{t-1} + i_t * \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
- **Output gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
- **Hidden state**: $h_t = o_t * \tanh(C_t)$

### Key Components

1. **RNN**
   - Processes sequences step by step
   - Maintains hidden state
   - Shares weights across time steps
   - Suffers from vanishing gradients

2. **LSTM (Long Short-Term Memory)**
   - Solves vanishing gradient problem
   - Uses gates to control information flow
   - Can remember long-term dependencies
   - More complex than RNN

3. **Gates in LSTM**
   - **Forget gate**: What to forget
   - **Input gate**: What to remember
   - **Output gate**: What to output

### How It Works

1. **Initialize**: Hidden state $h_0$
2. **Process sequence**: For each time step:
   - Compute hidden state (RNN) or cell/hidden (LSTM)
   - Generate output
3. **Backpropagation**: Through time (BPTT)
4. **Update weights**: Gradient descent

### Key Hyperparameters

- **hidden_size**: Number of hidden units
- **num_layers**: Number of RNN/LSTM layers
- **sequence_length**: Length of input sequences
- **dropout**: Regularization rate
- **bidirectional**: Process in both directions

### Advantages

- Handles variable-length sequences
- Can model temporal dependencies
- Shares parameters across time
- Good for sequential data
- LSTM solves vanishing gradients

### Limitations

- Slow training (sequential processing)
- Hard to parallelize
- May forget long-term dependencies (RNN)
- Computationally expensive
- Requires careful initialization


## Implementation

Let's implement RNNs and LSTMs for sequence prediction.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Libraries imported successfully!")


In [ ]:
# Create synthetic time series data
def generate_sine_wave(n_samples=1000, noise_level=0.1):
    """Generate sine wave time series"""
    t = np.linspace(0, 4*np.pi, n_samples)
    data = np.sin(t) + noise_level * np.random.randn(n_samples)
    return data

# Generate data
data = generate_sine_wave(n_samples=1000, noise_level=0.1)
data = data.reshape(-1, 1)

# Normalize
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

print(f"Time series data shape: {data_scaled.shape}")
print(f"Data range: [{data_scaled.min():.3f}, {data_scaled.max():.3f}]")

# Plot
plt.figure(figsize=(12, 4))
plt.plot(data_scaled[:200])
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Sample Time Series (First 200 Steps)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare sequences for RNN/LSTM
def create_sequences(data, seq_length=10):
    """Create sequences for time series prediction"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 20
X, y = create_sequences(data_scaled, seq_length)

# Split data
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)


In [ ]:
# Simple RNN Model
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, output_size=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # RNN forward
        out, hn = self.rnn(x, h0)
        
        # Take last output
        out = self.fc(out[:, -1, :])
        return out

# LSTM Model
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, output_size=1):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM forward
        out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # Take last output
        out = self.fc(out[:, -1, :])
        return out

print("RNN and LSTM models defined!")


## Training LSTM

Let's train the LSTM model.


In [ ]:
# Train LSTM
model_lstm = SimpleLSTM(input_size=1, hidden_size=32, num_layers=1, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

num_epochs = 50
train_losses = []

for epoch in range(num_epochs):
    model_lstm.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model_lstm(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: Loss = {loss.item():.6f}")

print("\nTraining complete!")


## Evaluation and Comparison

Let's compare RNN and LSTM performance.


In [ ]:
# Train RNN for comparison
model_rnn = SimpleRNN(input_size=1, hidden_size=32, num_layers=1, output_size=1).to(device)
optimizer_rnn = optim.Adam(model_rnn.parameters(), lr=0.001)

rnn_losses = []
for epoch in range(num_epochs):
    model_rnn.train()
    optimizer_rnn.zero_grad()
    outputs = model_rnn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer_rnn.step()
    rnn_losses.append(loss.item())

# Evaluate both models
model_lstm.eval()
model_rnn.eval()

with torch.no_grad():
    lstm_pred = model_lstm(X_test_tensor).cpu().numpy()
    rnn_pred = model_rnn(X_test_tensor).cpu().numpy()

# Calculate MSE
lstm_mse = mean_squared_error(y_test, lstm_pred)
rnn_mse = mean_squared_error(y_test, rnn_pred)

print("Model Comparison:")
print(f"  LSTM Test MSE: {lstm_mse:.6f}")
print(f"  RNN Test MSE: {rnn_mse:.6f}")

# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='LSTM', alpha=0.7)
plt.plot(rnn_losses, label='RNN', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: LSTM vs RNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Visualization

Let's visualize the predictions.


In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# LSTM predictions
axes[0].plot(y_test[:200], label='True', alpha=0.7, linewidth=2)
axes[0].plot(lstm_pred[:200], label='LSTM Prediction', alpha=0.7, linestyle='--')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Value')
axes[0].set_title('LSTM Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RNN predictions
axes[1].plot(y_test[:200], label='True', alpha=0.7, linewidth=2)
axes[1].plot(rnn_pred[:200], label='RNN Prediction', alpha=0.7, linestyle='--')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Value')
axes[1].set_title('RNN Predictions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the models.


In [ ]:
# Assertions
assert lstm_mse < 0.1, "LSTM should learn the pattern"
assert rnn_mse < 0.2, "RNN should learn the pattern"
print("\n✓ Validation checks passed")

print("\nNote: LSTM typically performs better on longer sequences")
print("due to its ability to remember long-term dependencies.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **RNN Basics**
   - Processes sequences step by step
   - Maintains hidden state
   - Shares weights across time
   - Suffers from vanishing gradients

2. **LSTM (Long Short-Term Memory)**
   - Solves vanishing gradient problem
   - Uses gates to control information
   - Can remember long-term dependencies
   - More complex but more powerful

3. **Key Components**
   - **Hidden state**: Carries information
   - **Gates**: Control information flow (LSTM)
   - **Cell state**: Long-term memory (LSTM)
   - **Sequence length**: How far back to look

4. **Applications**
   - Time series prediction
   - Natural language processing
   - Speech recognition
   - Sequence-to-sequence tasks

### When to Use RNNs and LSTMs

✅ **Good for:**
- Sequential data
- Time series prediction
- Natural language processing
- When order matters
- Variable-length sequences
- Need to remember past context

❌ **Not ideal for:**
- Tabular data (use MLPs)
- Image data (use CNNs)
- Very long sequences (use Transformers)
- When parallelization is critical
- Real-time applications (can be slow)

### Next Steps

- Explore **GRU (Gated Recurrent Unit)** as LSTM alternative
- Try **Bidirectional RNNs/LSTMs**
- Apply to **NLP tasks** (text generation, sentiment)
- Use **Attention mechanisms** for better performance
- Experiment with **Transformer models** for long sequences
